In [7]:
# ============================================================
# CELL 1 - Load Datasets
# ============================================================

import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/owid-co2-data.csv"
ENERGY_DATA_PATH = "../data/raw/owid-energy-data.csv"

df = pd.read_csv(DATA_PATH)
energy_df = pd.read_csv(ENERGY_DATA_PATH)

print("=" * 60)
print("DATASET LOADING")
print("=" * 60)
print("CO₂ dataset:", df.shape)
print("Energy dataset:", energy_df.shape)

DATASET LOADING
CO₂ dataset: (50411, 79)
Energy dataset: (23377, 130)


In [8]:
# ============================================================
# CELL 2 - Merge Audit
# ============================================================

print("=" * 60)
print("MERGE AUDIT")
print("=" * 60)

common_locations = set(df["country"]) & set(energy_df["country"])
common_years = set(df["year"]) & set(energy_df["year"])

print("Common locations:", len(common_locations))
print("Common years:", len(common_years))
print("Common year range:", min(common_years), "to", max(common_years))

merged = df.merge(
    energy_df,
    on=["country", "year"],
    how="inner",
    suffixes=("_co2", "_energy")
)

print("\nMerged shape:", merged.shape)
print("Duplicate country-year rows:", merged.duplicated(["country", "year"]).sum())

display(merged[["country", "year"]].head(10))

MERGE AUDIT
Common locations: 219
Common years: 125
Common year range: 1900 to 2024

Merged shape: (18063, 207)
Duplicate country-year rows: 0


,country,year
0,Afghanistan,1900
1,Afghanistan,1901
2,Afghanistan,1902
3,Afghanistan,1903
4,Afghanistan,1904
5,Afghanistan,1905
6,Afghanistan,1906
7,Afghanistan,1907
8,Afghanistan,1908
9,Afghanistan,1909


In [9]:
# ============================================================
# CELL 3 - P_132 Variable Selection & Missingness
# ============================================================

candidate_columns = [
    "country", "year", "iso_code_co2", "population_co2", "gdp_co2",
    "co2", "co2_per_capita", "co2_per_gdp",
    "primary_energy_consumption_co2",
    "coal_co2", "oil_co2", "gas_co2",
    "population_energy", "gdp_energy",
    "primary_energy_consumption_energy",
    "renewables_consumption", "renewables_share_energy",
    "renewables_electricity", "renewables_share_elec",
    "solar_consumption", "solar_share_energy",
    "wind_consumption", "wind_share_energy",
    "hydro_consumption", "hydro_share_energy",
    "fossil_fuel_consumption", "fossil_share_energy"
]

available_columns = [col for col in candidate_columns if col in merged.columns]

missing_summary = pd.DataFrame({
    "missing_count": merged[available_columns].isna().sum(),
    "missing_percentage": merged[available_columns].isna().mean().mul(100).round(2)
}).sort_values("missing_percentage", ascending=False)

print("=" * 60)
print("P_132 VARIABLES & MISSINGNESS")
print("=" * 60)
print("Available candidate variables:", len(available_columns))
print("\nCandidate variables:")
print(available_columns)
print("\nMissing-value summary:")
display(missing_summary)

P_132 VARIABLES & MISSINGNESS
Available candidate variables: 27

Candidate variables:
['country', 'year', 'iso_code_co2', 'population_co2', 'gdp_co2', 'co2', 'co2_per_capita', 'co2_per_gdp', 'primary_energy_consumption_co2', 'coal_co2', 'oil_co2', 'gas_co2', 'population_energy', 'gdp_energy', 'primary_energy_consumption_energy', 'renewables_consumption', 'renewables_share_energy', 'renewables_electricity', 'renewables_share_elec', 'solar_consumption', 'solar_share_energy', 'wind_consumption', 'wind_share_energy', 'hydro_consumption', 'hydro_share_energy', 'fossil_fuel_consumption', 'fossil_share_energy']

Missing-value summary:


,missing_count,missing_percentage
wind_share_energy,12944,71.66
fossil_share_energy,12944,71.66
renewables_share_energy,12944,71.66
solar_share_energy,12944,71.66
hydro_share_energy,12944,71.66
renewables_consumption,12938,71.63
solar_consumption,12938,71.63
wind_consumption,12938,71.63
hydro_consumption,12938,71.63
fossil_fuel_consumption,12938,71.63


In [12]:
# ============================================================
# CELL 4 - Detailed Coverage Audit
# ============================================================

core_columns = [
    "co2", "population_co2", "gdp_co2",
    "primary_energy_consumption_energy",
    "renewables_consumption", "renewables_share_energy"
]

print("=" * 60)
print("DETAILED COVERAGE AUDIT")
print("=" * 60)

year_coverage = merged.groupby("year")[core_columns].apply(lambda x: x.notna().mean() * 100)

print("Coverage for recent years:")
display(year_coverage.tail(31).round(2))

print("\nRenewable data availability by year:")
renewable_years = merged.groupby("year")["renewables_consumption"].apply(lambda x: x.notna().sum())
display(renewable_years.tail(31))

renewable_locations = merged.loc[merged["renewables_consumption"].notna(), "country"].nunique()
print("\nLocations with renewable-consumption data:", renewable_locations)

DETAILED COVERAGE AUDIT
Coverage for recent years:


,co2,population_co2,gdp_co2,primary_energy_consumption_energy,renewables_consumption,renewables_share_energy
year,,,,,,
1994,100.0,99.53,75.70,99.07,42.06,42.06
1995,100.0,99.53,75.70,99.07,42.06,42.06
1996,100.0,99.53,75.70,99.07,42.06,42.06
1997,100.0,99.53,75.81,99.07,41.86,41.86
1998,100.0,99.53,75.81,99.07,41.86,41.86
1999,100.0,99.53,75.81,99.07,41.86,41.86
2000,100.0,99.54,75.93,98.61,41.67,41.67
2001,100.0,99.54,75.46,98.61,41.67,41.67
2002,100.0,99.54,75.46,98.61,41.67,41.67



Renewable data availability by year:


year
1994    90
1995    90
1996    90
1997    90
1998    90
1999    90
2000    90
2001    90
2002    90
2003    90
2004    90
2005    90
2006    90
2007    90
2008    90
2009    90
2010    90
2011    90
2012    90
2013    90
2014    90
2015    90
2016    90
2017    90
2018    90
2019    90
2020    90
2021    90
2022    90
2023    90
2024    90
Name: renewables_consumption, dtype: int64


Locations with renewable-consumption data: 90


In [13]:
# ============================================================
# CELL 5 - Country-Level Coverage
# ============================================================

print("=" * 60)
print("COUNTRY-LEVEL COVERAGE")
print("=" * 60)

country_data = merged[merged["iso_code_co2"].notna()].copy()

country_coverage = country_data.groupby("country")[core_columns].apply(lambda x: x.notna().mean() * 100)
country_coverage["overall_coverage"] = country_coverage.mean(axis=1)

print("ISO-coded locations:", country_data["country"].nunique())

print("\nBest-covered locations:")
display(country_coverage.sort_values("overall_coverage", ascending=False).head(30).round(2))

print("\nLocations with at least 80% coverage for all core variables:")
good_countries = country_coverage.index[(country_coverage[core_columns] >= 80).all(axis=1)]
print("Count:", len(good_countries))
print(good_countries.tolist())

COUNTRY-LEVEL COVERAGE
ISO-coded locations: 206

Best-covered locations:


,co2,population_co2,gdp_co2,primary_energy_consumption_energy,renewables_consumption,renewables_share_energy,overall_coverage
country,,,,,,,
Cyprus,100.0,100.0,96.67,100.0,100.0,100.0,99.44
Finland,100.0,100.0,96.67,100.0,100.0,100.0,99.44
Singapore,100.0,100.0,96.67,100.0,100.0,100.0,99.44
Luxembourg,100.0,100.0,96.67,100.0,100.0,100.0,99.44
Hong Kong,100.0,100.0,96.67,100.0,100.0,100.0,99.44
Iceland,100.0,100.0,96.67,100.0,100.0,100.0,99.44
Sri Lanka,100.0,100.0,96.67,100.0,100.0,100.0,99.44
Kazakhstan,100.0,100.0,95.00,100.0,100.0,100.0,99.17
Latvia,100.0,100.0,95.00,100.0,100.0,100.0,99.17



Locations with at least 80% coverage for all core variables:
Count: 20
['Azerbaijan', 'Belarus', 'Croatia', 'Cyprus', 'Czechia', 'Estonia', 'Finland', 'Hong Kong', 'Iceland', 'Kazakhstan', 'Latvia', 'Lithuania', 'Luxembourg', 'North Macedonia', 'Russia', 'Singapore', 'Slovenia', 'Sri Lanka', 'Turkmenistan', 'Uzbekistan']


In [14]:
# ============================================================
# CELL 6 - Compare Candidate Modelling Periods
# ============================================================

print("=" * 60)
print("CANDIDATE MODELLING PERIODS")
print("=" * 60)

periods = [(1990, 2022), (1995, 2022), (2000, 2022), (2000, 2023)]

for start, end in periods:
    period_df = merged[(merged["year"] >= start) & (merged["year"] <= end)]
    coverage = period_df[core_columns].notna().mean() * 100
    valid_countries = period_df.groupby("country")[core_columns].apply(lambda x: x.notna().mean() * 100)
    good = valid_countries.index[(valid_countries >= 80).all(axis=1)]
    print(f"\n{start}-{end}:")
    print("Rows:", len(period_df), "| Countries:", period_df["country"].nunique())
    print("Countries with ≥80% coverage for all core variables:", len(good))
    print("Average coverage:")
    print(coverage.round(2))

CANDIDATE MODELLING PERIODS

1990-2022:
Rows: 7144 | Countries: 219
Countries with ≥80% coverage for all core variables: 79
Average coverage:
co2                                  99.96
population_co2                       99.54
gdp_co2                              75.41
primary_energy_consumption_energy    99.19
renewables_consumption               41.57
renewables_share_energy              41.57
dtype: float64

1995-2022:
Rows: 6090 | Countries: 219
Countries with ≥80% coverage for all core variables: 79
Average coverage:
co2                                  100.00
population_co2                        99.54
gdp_co2                               75.37
primary_energy_consumption_energy     99.21
renewables_consumption                41.38
renewables_share_energy               41.38
dtype: float64

2000-2022:
Rows: 5017 | Countries: 219
Countries with ≥80% coverage for all core variables: 79
Average coverage:
co2                                  100.00
population_co2                    

In [15]:
# ============================================================
# CELL 7 - Renewable-Data Country Coverage
# ============================================================

print("=" * 60)
print("RENEWABLE-DATA COUNTRY COVERAGE")
print("=" * 60)

renewable_countries = sorted(merged.loc[merged["renewables_consumption"].notna(), "country"].unique())

print("Countries with renewable-consumption data:", len(renewable_countries))
print("\nCountries:")
print(renewable_countries)

RENEWABLE-DATA COUNTRY COVERAGE
Countries with renewable-consumption data: 90

Countries:
['Africa', 'Algeria', 'Argentina', 'Asia', 'Australia', 'Austria', 'Azerbaijan', 'Bangladesh', 'Belarus', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Ecuador', 'Egypt', 'Estonia', 'Europe', 'European Union (27)', 'Finland', 'France', 'Germany', 'Greece', 'High-income countries', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Japan', 'Kazakhstan', 'Kuwait', 'Latvia', 'Lithuania', 'Lower-middle-income countries', 'Luxembourg', 'Malaysia', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'North America', 'North Macedonia', 'Norway', 'Oceania', 'Oman', 'Pakistan', 'Peru', 'Philippines', 'Poland', 'Portugal', 'Qatar', 'Romania', 'Russia', 'Saudi Arabia', 'Singapore', 'Slovakia', 'Slovenia', 'South Africa', 'South America', 'South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Sw

In [16]:
# ============================================================
# CELL 8 - Preliminary Modelling Dataset
# ============================================================

START_YEAR = 2000
END_YEAR = 2022

model_base = merged[(merged["year"] >= START_YEAR) & (merged["year"] <= END_YEAR)].copy()

country_coverage = model_base.groupby("country")[core_columns].apply(lambda x: x.notna().mean() * 100)
selected_countries = country_coverage.index[(country_coverage >= 80).all(axis=1)]

model_data = model_base[model_base["country"].isin(selected_countries)].copy()

print("=" * 60)
print("PRELIMINARY MODELLING DATASET")
print("=" * 60)
print("Period:", START_YEAR, "-", END_YEAR)
print("Selected countries:", len(selected_countries))
print("Rows:", len(model_data))
print("Columns:", model_data.shape[1])

print("\nSelected countries:")
print(selected_countries.tolist())

print("\nMissingness in selected dataset:")
display(model_data[core_columns].isna().mean().mul(100).round(2))

PRELIMINARY MODELLING DATASET
Period: 2000 - 2022
Selected countries: 79
Rows: 1817
Columns: 207

Selected countries:
['Algeria', 'Argentina', 'Australia', 'Austria', 'Azerbaijan', 'Bangladesh', 'Belarus', 'Belgium', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China', 'Colombia', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Ecuador', 'Egypt', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Japan', 'Kazakhstan', 'Kuwait', 'Latvia', 'Lithuania', 'Luxembourg', 'Malaysia', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'North Macedonia', 'Norway', 'Oman', 'Pakistan', 'Peru', 'Philippines', 'Poland', 'Portugal', 'Qatar', 'Romania', 'Russia', 'Saudi Arabia', 'Singapore', 'Slovakia', 'Slovenia', 'South Africa', 'South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Switzerland', 'Taiwan', 'Thailand', 'Trinidad and Tobago', 'Turkey', 'Turkmenistan', 'Ukraine', 'United Arab Emirates', 'United 

co2                                  0.0
population_co2                       0.0
gdp_co2                              0.0
primary_energy_consumption_energy    0.0
renewables_consumption               0.0
renewables_share_energy              0.0
dtype: float64

In [18]:
# ============================================================
# CELL 9 - Final Variable Coverage Audit & Save
# ============================================================

final_candidates = [
    "country", "year", "iso_code_co2", "population_co2", "gdp_co2", "co2",
    "coal_co2", "oil_co2", "gas_co2", "cement_co2",
    "primary_energy_consumption_energy", "fossil_fuel_consumption",
    "fossil_share_energy", "renewables_consumption", "renewables_share_energy",
    "renewables_electricity", "renewables_share_elec",
    "solar_consumption", "solar_share_energy", "wind_consumption", "wind_share_energy",
    "hydro_consumption", "hydro_share_energy"
]

final_columns = [col for col in final_candidates if col in model_data.columns]
final_data = model_data[final_columns].copy()

coverage = final_data.notna().mean().mul(100).round(2).sort_values()

print("=" * 60)
print("FINAL VARIABLE COVERAGE AUDIT")
print("=" * 60)
print("Selected columns:", len(final_columns))
print("Rows:", len(final_data))
print("\nVariable coverage:")
display(coverage.to_frame("coverage_percentage"))

OUTPUT_PATH = "../data/processed/cleaned_data.csv"
final_data.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)
print("Final dataset shape:", final_data.shape)

FINAL VARIABLE COVERAGE AUDIT
Selected columns: 23
Rows: 1817

Variable coverage:


,coverage_percentage
coal_co2,97.03
gas_co2,98.73
cement_co2,98.73
country,100.00
population_co2,100.00
gdp_co2,100.00
iso_code_co2,100.00
year,100.00
oil_co2,100.00
co2,100.00



Saved: ../data/processed/cleaned_data.csv
Final dataset shape: (1817, 23)
